In [1]:
suppressMessages(library(dplyr))
suppressMessages(library(ggplot2))
suppressMessages(library(cowplot))
suppressMessages(library(patchwork))
suppressMessages(library(ComplexHeatmap))
suppressMessages(library(tidyverse))
suppressMessages(library(circlize))
suppressMessages(library(ArchR))
suppressMessages(library(scales))

Warning message:
“package ‘matrixStats’ was built under R version 4.3.3”


In [2]:
input_dir <- "../../results/03_eval_simulate_data/04_spatial_pattern_clustering"

In [3]:
out_dir <- "../../results/03_eval_simulate_data/05_spatial_pattern_clustering"

if(!dir.exists(out_dir)){
    dir.create(out_dir)
}


In [4]:
dataset_list <- c('01_10x_Visium_mouse_brain',
                '02_10x_Visium_Human_Heart_MI',
                '03_10x_Visium_Human_Colorectal_Cancer',
                '04_10x_Visium_Human_Heart',
                '05_10x_Visium_Mouse_Embryo',
                '06_10x_Visium_Mouse_Olfactory_Bulb',
                '07_10x_Visium_Human_Breast_Cancer',
                '08_10x_Visium_Human_Lymph_Node',
                '09_10x_Visium_Human_Prostate',
                '10_10x_Visium_Human_Prostate_Cancer',
                '11_10x_Visium_Human_Cerebellum',
                '12_10x_Visium_Mouse_Kidney',
                '13_10x_Visium_Human_Lung_Cancer',
                '14_10x_Visium_Human_Brain_Cancer',
                '15_10x_Visium_Human_Kidney',
                '16_10x_Visium_Human_Intestine_Cancer',
                '17_10x_Visium_Human_Skin_Melanoma',
                '18_10x_Visium_Human_Cervical_Cancer',
                '19_10x_Visium_Human_Breast_Cancer',
                '20_10x_Visium_Human_Heart_MI')


#dataset_list <- c('01_10x_Visium_mouse_brain')

In [5]:
options(repr.plot.height = 5, repr.plot.width = 4.5)


for(dataset in dataset_list){
    df <- read.csv(glue::glue("{input_dir}/{dataset}.csv"),
                        row.names = 1)    
    
    # cluster genes 
    hc <- hclust(dist(df), method = "ward.D", members = NULL)
    df_cluster <- cutree(hc, k = 4)
    df_cluster <- as.data.frame(df_cluster)
    colnames(df_cluster) <- "cluster"
    df_cluster$cluster <- as.character(df_cluster$cluster)
    
    
    col_fun <- colorRamp2(seq(from = -0.5, to = 1, length = 9), 
                     RColorBrewer::brewer.pal(name = "YlGnBu", n = 9))

    ht <- Heatmap(as.matrix(df),
                  column_title = dataset,
                  cluster_columns = TRUE, 
                  cluster_rows = TRUE,
                  row_split = df_cluster$cluster, 
                  column_split = df_cluster$cluster,
                  col = col_fun,
                  row_gap = unit(0.5, "mm"),
                  column_gap = unit(0.5, "mm"),
                  show_column_names = FALSE,
                  show_row_names = FALSE,
                  show_parent_dend_line = FALSE,
                  cluster_row_slices = FALSE,
                  cluster_column_slices = FALSE,
                  heatmap_legend_param = list(title = "Correlation",
                                              title_position = "topcenter",
                                              direction = "horizontal")

                 )

    pdf(glue::glue("{out_dir}/{dataset}.pdf"), height = 6, width = 6)
    draw(ht, heatmap_legend_side = "bottom")
    dev.off()
}